In [1]:
import sys
import os

# For notebooks, use getcwd(); for scripts, use __file__
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Running in a notebook
    notebook_dir = os.getcwd()

sys.path.insert(0, notebook_dir)

from agents.orchestrator import analyze_user_input, update_state_from_analysis
from agents.specialist import run_activities_specialist, run_logistics_specialist
from agents.verifier import verify_and_format_itinerary, format_complete_itinerary
from state import TravelState


In [ ]:
state = TravelState()
task_json = {
  "task_id": "task_123",
  "intent": "update_constraints",
  "updated_constraints": {
    "origin": "BOS",
    "destination": "SEA",
    "start_date": "2026-05-05",
    "end_date": "2026-05-12"
  },
  "delegation": "logistics",
  "response_to_user": "I will search for flights...",
}
user_input = None #"want to change my trip to Seattle from May 6th to May 13th"

analysis = analyze_user_input(user_input, state, task_json=task_json)
state = update_state_from_analysis(state, analysis)
delegation = analysis.get("delegation", "none")

In [ ]:
 # 3. Delegation
if delegation == "none":
    response = analysis.get(
        "response_to_user",
        "I'm still missing some info to start planning.",
    )
    print(f"\nNomad: {response}")
    state.messages.append({"role": "assistant", "content": response})

In [ ]:

# If we reach here, we need to run specialists
draft_components = []
all_search_results = {
    "flights": [],
    "hotels": [],
    "activities": [],
}
# Accumulate specialist contexts for verifier
specialist_contexts = {
    "logistics": None,
    "activities": None,
}
constraints_str = state.constraints.model_dump_json(indent=2)

if delegation in ["logistics", "both"]:
    print("\n[Specialist - Logistics] Searching for flights & hotels...")
    try:
        # NEW: Unpack all 3 return values
        logistics_draft, logistics_searches, logistics_context = run_logistics_specialist(
            constraints_str, 
            task_id=state.task_id
        )
        draft_components.append("--- LOGISTICS ---\n" + logistics_draft)
        
        # Accumulate search results
        all_search_results["flights"].extend(logistics_searches.get("flights", []))
        all_search_results["hotels"].extend(logistics_searches.get("hotels", []))
        
        # Save specialist context for verifier
        specialist_contexts["logistics"] = logistics_context
        
        print(f"\n[Logistics Specialist Summary]")
        print(f"  Search Coverage: {logistics_context['search_coverage']}")
        print(f"  Total Searches: {logistics_context['total_searches']}")
        print(f"  Tool Calls: {len(logistics_context['tool_calls_summary'])}")
        
    except Exception as e:
        print(f"[Specialist Error] {e}")
        import traceback
        traceback.print_exc()


In [ ]:
print("\n[Verifier] Checking itinerary against constraints...")
full_draft = "\n\n".join(draft_components)

print(f"\n[Debug] Search Results Found:")
print(f"  - Flights: {len(all_search_results['flights'])} items")
print(f"  - Hotels: {len(all_search_results['hotels'])} items")
print(f"  - Activities: {len(all_search_results['activities'])} items")

try:
    verification = verify_and_format_itinerary(
        full_draft, 
        constraints_str,
        task_id=state.task_id,
        search_results=all_search_results  # Pass search results to verifier
    )

    if verification.get("is_valid"):
        final_response = verification.get(
            "final_message_to_user", "Here is your plan!"
        )
        print("\nNomad:\n")
        print(final_response)
        state.messages.append({"role": "assistant", "content": final_response})
    else:
        issues = verification.get("issues", [])
        print(
            f"\nNomad: I built a plan, but it violates some constraints:\n{issues}"
        )
        print("I will need to revise. Let me know how you'd like to adjust.")

except Exception as e:
    print(f"Error during verification: {e}")

In [ ]:
verification

In [ ]:
# 5. Save Agent Result to Repository (Independent)
# ===================================================
# 保存完整的 verification 结果（含 constraints），方便后续评估
from tools.plan_repository import PlanRepository

print("\n" + "="*70)
print("[SAVE] Saving agent output to repository...")
print("="*70)

if verification.get("is_valid"):
    plan_to_save = verification.get("itinerary", verification)
    
    # Save to repository with metadata
    repo = PlanRepository(base_dir="plans")
    saved_path = repo.save_plan(
        plan=plan_to_save, 
        task_id=state.task_id,
        save_metadata=True
    )
    
    print(f"✓ Plan saved to: {saved_path}")
    print(f"  Includes: itinerary + constraints (for evaluator)")
    
    # Show repository status
    all_plans = repo.get_all_plans()
    print(f"\n[REPOSITORY STATUS]")
    print(f"  Total plans saved: {len(all_plans)}")
    print(f"  Task IDs: {all_plans}")
    print(f"\n💡 Tip: Run 'BATCH EVALUATOR' cell to evaluate all saved plans at once")
    
else:
    print(f"✗ Cannot save invalid plan")
    print(f"  Issues: {verification.get('issues', [])}")

In [2]:
# 6. BATCH EVALUATOR - Evaluate All Saved Plans at Once
# =========================================================
# 统一评估所有之前保存的 plans
import importlib
import tools.plan_schema, tools.evaluator, tools.auto_evaluator
importlib.reload(tools.plan_schema)
importlib.reload(tools.evaluator)
importlib.reload(tools.auto_evaluator)
from tools.auto_evaluator import AutoEvaluator
import json
from pathlib import Path

print("\n" + "="*70)
print("[BATCH EVALUATOR] Evaluating all saved plans from repository")
print("="*70)

# Initialize auto evaluator
auto_eval = AutoEvaluator(plan_repo_dir="plans", output_dir="evaluations")

# Show what we're about to evaluate
repo_plans = auto_eval.repo.get_all_plans()
print(f"\nFound {len(repo_plans)} plans to evaluate:")
for task_id in repo_plans:
    print(f"   - {task_id}")

print("\nEvaluating...")

# One-line evaluation: read all plans -> evaluate -> generate report
results, report_file = auto_eval.auto_evaluate_and_report(report_name="batch_evaluation")

print(f"\n[DONE] Report saved to: {report_file}")

# Display statistics
if results:
    scores = [r['overall_score'] for r in results.values() if 'overall_score' in r]
    
    if scores:
        print(f"\n[SUMMARY]")
        print(f"  Total evaluated: {len(scores)}")
        print(f"  Average score: {sum(scores)/len(scores):.1%}")
        print(f"  Best score: {max(scores):.1%}")
        print(f"  Worst score: {min(scores):.1%}")
    
    excellent = sum(1 for s in scores if s >= 0.8)
    acceptable = sum(1 for s in scores if 0.6 <= s < 0.8)
    needs_revision = sum(1 for s in scores if s < 0.6)
    
    print(f"\n[RATINGS]")
    print(f"  Excellent (>=0.8): {excellent}")
    print(f"  Acceptable (0.6-0.8): {acceptable}")
    print(f"  Needs Revision (<0.6): {needs_revision}")
    
    # Detail per plan
    for task_id, result in results.items():
        if 'overall_score' in result:
            print(f"\n[{task_id}]")
            print(f"  Overall: {result['overall_score']:.1%}")
            print(f"  Schema:  {result.get('schema_compliance', 0):.1%}")
            print(f"  CSR:     {result.get('csr_score', 0):.1%}")
            print(f"  Tools:   {result.get('tool_accuracy', 0):.1%}")
            if result.get('constraint_breakdown'):
                for c, met in result['constraint_breakdown'].items():
                    print(f"    {'Y' if met else 'N'} {c}")
            if result.get('conflict_report', {}).get('warnings'):
                for w in result['conflict_report']['warnings']:
                    print(f"    ! {w}")


[BATCH EVALUATOR] Evaluating all saved plans from repository

Found 1 plans to evaluate:
   - task_123

Evaluating...

[AutoEvaluator] Evaluating 1 plans...
[AutoEvaluator] Evaluating task_123... ✓ Loaded plan from repository: task_123
✓ Score: 100.0%

EVALUATION SUMMARY
  ✓ EXCELLENT     task_123             100.0%

----------------------------------------------------------------------
TOTALS                             1
  Excellent:     1
  Acceptable:    0
  Needs Revision: 0

Average Score: 100.0%
Best Score:    100.0%
Worst Score:   100.0%

✓ Report saved: evaluations\batch_evaluation.json

[DONE] Report saved to: evaluations\batch_evaluation.json

[SUMMARY]
  Total evaluated: 1
  Average score: 100.0%
  Best score: 100.0%
  Worst score: 100.0%

[RATINGS]
  Excellent (>=0.8): 1
  Acceptable (0.6-0.8): 0
  Needs Revision (<0.6): 0

[task_123]
  Overall: 100.0%
  Schema:  100.0%
  CSR:     100.0%
  Tools:   100.0%
    Y Origin: BOS
    Y Destination: SEA
    Y Start_date: 2026-05-

In [ ]:
# DEBUG: Direct evaluation test
import importlib, tools.plan_schema, tools.evaluator, tools.plan_repository
importlib.reload(tools.plan_schema)
importlib.reload(tools.evaluator)
from tools.evaluator import NomadEvaluator
from tools.plan_repository import PlanRepository

repo = PlanRepository(base_dir="plans")
plan = repo.load_plan("task_123")

ev = NomadEvaluator()
result = ev.evaluate(agent_output=plan, task_id="task_123")

print("Overall:", result['overall_score'])
print("Schema:", result['schema_compliance'])
print("CSR:", result['csr_score'])
print("Tools:", result['tool_accuracy'])
print("Validity:", result['itinerary_validity'])
print()
print("Constraints:")
for c, met in result['constraint_breakdown'].items():
    print(f"  {'Y' if met else 'N'} {c}")
print()
print("Conflicts:", result['conflict_report'])